# Iowa Liquor Promotion Experiment Proposal
### Portfolio refresh of a 2019 Thinkful analysis

This notebook is a presentation and methodology refresh of the original `Iowa_Liquor_Store_AB_Test_Proposal.ipynb`. The original notebook remains unchanged.

The 2019 project explored Iowa liquor order data, identified high-value products within Casey's stores, and proposed an in-store promotion test. This portfolio version preserves that analytical history while correcting the interpretation of the public dataset and redesigning the proposed experiment to meet modern A/B testing standards.

## 1. Business Question

**Exploratory question:** Which products account for meaningful order activity across Casey's locations, and where is product penetration comparatively low?

**Experiment question:** If selected Casey's stores increase the visibility of a target product, does that intervention increase customer purchases relative to comparable control stores?

The public Iowa data are useful for baseline analysis and store/product selection. A true promotion experiment would require contemporaneous treatment and control outcomes, preferably Casey's point-of-sale data.

## 2. Data Source and Historical Scope

The source is the **Iowa Liquor Sales** public dataset maintained by the State of Iowa. It contains liquor purchase orders made by Iowa Class E liquor licensees. It is not a consumer point-of-sale dataset.

The preserved 2019 notebook used a local snapshot of the old combined Iowa dataset containing roughly 12.59 million rows. Executed outputs show the snapshot extending through **October 31, 2017**.

To avoid downloading today's multi-gigabyte statewide table, this portfolio version uses `scripts/download_iowa_caseys_subset.py` to request only Casey's records from 2012 through October 31, 2017 and only the columns needed for this project.

### Important interpretation correction

In the original notebook, the derived field `Profit` was calculated as:

`Sale (Dollars) - State Bottle Cost × Bottles Sold`

Iowa's metadata define `State Bottle Cost` as the amount the Alcoholic Beverages Division paid per bottle and `Sale (Dollars)` as the total amount charged to the licensee for the order. Therefore, this difference is **not store profit**. It is closer to a state/wholesale gross-margin measure on the order.

This refresh retains the old calculation only as a provenance check and does not use it as evidence of Casey's profitability.

## 3. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["font.size"] = 11

## 4. Load the Historical Casey's Subset

Run `python scripts/download_iowa_caseys_subset.py` from the repository root first. The downloader queries Iowa's API directly and saves a much smaller Casey's-only CSV.

In [ ]:
DATA_CANDIDATES = [
    Path("data/iowa_caseys_2012_to_2017_10_31.csv"),
    Path("../../data/iowa_caseys_2012_to_2017_10_31.csv"),
]
DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Historical Casey's subset not found. Run "
        "python scripts/download_iowa_caseys_subset.py from the repository root."
    )

casey = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Loaded {len(casey):,} Casey's order lines from {DATA_PATH}")

## 5. Preparation and Data Quality Checks

In [ ]:
casey["date"] = pd.to_datetime(casey["date"], errors="coerce")

numeric_columns = [
    "pack", "bottle_volume_ml", "state_bottle_cost",
    "state_bottle_retail", "sale_bottles", "sale_dollars", "sale_liters",
]
for column in numeric_columns:
    casey[column] = pd.to_numeric(casey[column], errors="coerce")

casey["itemno"] = casey["itemno"].astype("string")
casey["name"] = casey["name"].astype("string")

print("Date range:", casey["date"].min(), "to", casey["date"].max())
print("Unique store names:", casey["name"].nunique())
print("Missing sale dollars:", casey["sale_dollars"].isna().sum())

In [ ]:
# Iowa describes sale_dollars as state_bottle_retail × sale_bottles.
# Historical records can contain inconsistencies, so inspect the relationship before
# using any derived margin measure.
casey["expected_order_value"] = (
    casey["state_bottle_retail"] * casey["sale_bottles"]
)
casey["order_value_gap"] = casey["sale_dollars"] - casey["expected_order_value"]
casey["order_value_consistent"] = casey["order_value_gap"].abs() <= 0.02

quality_summary = pd.Series({
    "rows": len(casey),
    "consistent_rows": int(casey["order_value_consistent"].sum()),
    "inconsistent_rows": int((~casey["order_value_consistent"]).sum()),
    "consistent_share": casey["order_value_consistent"].mean(),
})
quality_summary

The original notebook included negative values in its derived `Profit` field. Before interpreting any margin-like calculation, this refresh explicitly checks whether the source fields satisfy the relationship described in Iowa's metadata. For the portfolio analysis below, direct order measures such as `sale_dollars`, `sale_liters`, and store penetration are more defensible than calling the derived difference retailer profit.

## 6. Reproduce the Original Derived Metric for Provenance

In [ ]:
casey["state_acquisition_cost"] = (
    casey["state_bottle_cost"] * casey["sale_bottles"]
)
casey["derived_wholesale_margin"] = (
    casey["sale_dollars"] - casey["state_acquisition_cost"]
)

historical_margin_by_item = (
    casey.groupby(["itemno", "category_name"], dropna=False)
    ["derived_wholesale_margin"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)
historical_margin_by_item

### Preserved 2019 output

Using the original calculation, the notebook selected item numbers **11788**, **11776**, and **35918** as its three leading products. Historical Iowa product records identify these as Black Velvet Canadian whisky variants and Five O'Clock Vodka.

This is preserved as an historical result. The refreshed analysis below does not equate the derived metric with store profit.

## 7. Portfolio Analysis: Order Activity and Product Penetration

In [ ]:
total_stores = casey["name"].nunique()

product_summary = (
    casey.groupby(["itemno", "im_desc"], dropna=False)
    .agg(
        order_value=("sale_dollars", "sum"),
        liters_ordered=("sale_liters", "sum"),
        order_lines=("invoice_line_no", "count"),
        stores=("name", "nunique"),
    )
    .reset_index()
)
product_summary["store_penetration"] = product_summary["stores"] / total_stores

product_summary.sort_values("order_value", ascending=False).head(15)

In [ ]:
top_products = product_summary.nlargest(15, "order_value").copy()
top_products = top_products.sort_values("order_value")

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top_products["im_desc"].fillna(top_products["itemno"]), top_products["order_value"])
ax.set_title("Top Casey's Products by Historical Order Value")
ax.set_xlabel("Order value paid by stores (USD)")
ax.set_ylabel("Product")
plt.tight_layout()
plt.show()

Order value measures how much Casey's locations paid for liquor orders in the public Iowa system. It should not be described as customer revenue. Store penetration complements order value by showing how broadly each product appears across locations.

In [ ]:
historical_target_items = {"11788", "11776", "35918"}
target_item_rows = casey[casey["itemno"].isin(historical_target_items)].copy()

store_baseline = (
    casey.groupby("name")
    .agg(
        total_order_value=("sale_dollars", "sum"),
        total_liters=("sale_liters", "sum"),
        active_order_lines=("invoice_line_no", "count"),
    )
)
target_by_store = (
    target_item_rows.groupby("name")
    .agg(
        target_order_value=("sale_dollars", "sum"),
        target_liters=("sale_liters", "sum"),
        target_items=("itemno", "nunique"),
    )
)
store_baseline = store_baseline.join(target_by_store, how="left").fillna(0)
store_baseline["target_order_share"] = (
    store_baseline["target_order_value"] / store_baseline["total_order_value"]
)
store_baseline.sort_values(["target_items", "target_order_share"]).head(20)

A stronger targeting strategy would not simply choose stores with the lowest derived margin. It would identify stores with meaningful overall liquor activity but unusually low penetration or order share for the selected product, then match or stratify stores on baseline volume before random assignment.

## 8. What the Original Store Selection Produced

The preserved notebook's original metric ranked three Casey's locations as the lowest in the field it called `Profit`:

- Casey's General Store #3055 / Grundy Center: **-8,636.96**
- Casey's General Store #3050 / Council Bluffs: **-5,026.36**
- Casey's General Store #1706 / Winterset: **-3,942.68**

After removing the three selected products, the corresponding values became -11,074.41, -6,206.88, and -4,417.16. The original notebook used these differences to motivate promotion candidates.

Because the underlying metric was misinterpreted as retailer profit and because negative derived margins indicate data-quality or transaction-structure issues that require investigation, this portfolio version does **not** recommend those locations on that basis.

## 9. Redesigned Store-Level Experiment

### Hypothesis
Increasing in-store visibility for a selected target product will increase product-level unit sales per store-week relative to comparable stores without the merchandising change.

### Design

- **Experimental unit:** Casey's store. Randomize at the store level so customers within a store receive the same merchandising condition.
- **Treatment:** Increased visibility for one pre-selected product, such as an end-cap, shelf placement, signage, or another standardized merchandising intervention.
- **Control:** Business-as-usual placement.
- **Primary outcome:** Point-of-sale units sold per store-week for the target SKU.
- **Secondary outcomes:** Product revenue, retailer gross margin if available from internal data, basket attachment, and reorder volume.
- **Guardrails:** Stockouts, substitution away from close alternatives, and implementation compliance.
- **Assignment:** Stratify or match stores by baseline target-product volume, total liquor activity, geography, and store size; randomize within strata.
- **Pre-period:** Use several weeks of baseline data to improve precision and verify balance.
- **Test duration:** Pre-specify the duration based on expected traffic and a power calculation rather than stopping when an 'obvious' increase appears.
- **Analysis:** Estimate the intention-to-treat effect at the store level, ideally using a baseline-adjusted regression or difference-in-differences specification. Report confidence intervals and effect size, not only a p-value.

### Why historical Iowa data are not enough to measure the experiment

Historical data can help choose products, establish seasonality, and create balanced treatment/control groups. It cannot substitute for post-treatment outcomes. The 2019 rollout plan suggested that existing historical data meant no new data collection was necessary; this refresh corrects that point. A promotion effect can only be estimated after the intervention occurs and outcomes are observed for both treatment and control stores.

## 10. Limitations

- Iowa's public data describe licensee purchase orders, not end-consumer transactions.
- Historical store names contain formatting variations and should ideally be resolved using store numbers.
- Product/category labels changed over time. Item number is generally the stronger historical identifier.
- Some historical rows may not satisfy the current metadata relationship between bottles ordered, bottle retail price, and order value.
- The original derived `Profit` field should not be interpreted as Casey's retailer profit.
- The public data can support experiment planning, but a credible promotion test needs contemporaneous treatment/control outcomes.

## 11. Portfolio Takeaway

The original 2019 project demonstrated large-dataset cleaning, grouping, product/store segmentation, and experimental thinking. The portfolio refresh adds a more important professional skill: validating what the source fields actually mean before translating calculations into business conclusions.

The resulting recommendation is to use public Iowa order data for **baseline product and store analysis**, then evaluate a merchandising intervention with a **randomized store-level experiment using actual post-treatment outcomes**.

---

**Original notebook:** `Iowa_Liquor_Store_AB_Test_Proposal.ipynb`  
**Original work:** Thinkful Data Science coursework, 2019  
**Public data:** State of Iowa, Iowa Liquor Sales